# Coursera Course Dataset - Exploration

Dataset: `coursera_course_2024.csv` (6,645 rows x 14 columns)

In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv('coursera_course_2024.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
df.info()

## Missing Values

In [ ]:
total = len(df)
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / total * 100).round(1)
pd.DataFrame({'count': missing, 'pct': missing_pct.map(lambda x: f'{x}%')})

## Data Quality Issues

Some fields contain placeholder strings instead of NaN:

In [ ]:
# Placeholder values that should be treated as missing
print('=== rating ===')
print(f'  "Rating not found": {(df["rating"] == "Rating not found").sum()}')
print()
print('=== enrolled ===')
print(f'  "Enrollment number not found": {(df["enrolled"] == "Enrollment number not found").sum()}')
print()
print('=== Skills ===')
print(f'  "[]": {(df["Skills"] == "[]").sum()}')
print()
print('--- Effective missing (NaN + placeholder) ---')
effective_missing = {
    'Satisfaction Rate': df['Satisfaction Rate'].isnull().sum(),
    'Schedule': df['Schedule'].isnull().sum(),
    'num_reviews': df['num_reviews'].isnull().sum(),
    'enrolled': (df['enrolled'] == 'Enrollment number not found').sum(),
    'rating': (df['rating'] == 'Rating not found').sum(),
    'Skills (empty [])': (df['Skills'] == '[]').sum(),
    'Level': df['Level'].isnull().sum(),
}
for k, v in sorted(effective_missing.items(), key=lambda x: -x[1]):
    print(f'  {k}: {v} ({v/total*100:.1f}%)')

## Level Distribution

In [ ]:
df['Level'].value_counts(dropna=False).plot(kind='barh', title='Level Distribution')
print(df['Level'].value_counts(dropna=False))

## Rating Distribution

In [ ]:
# Exclude 'Rating not found'
valid_ratings = df[df['rating'] != 'Rating not found']['rating'].astype(float)
print(f'Valid ratings: {len(valid_ratings)} / {total}')
print(valid_ratings.describe())
print()
valid_ratings.hist(bins=20, edgecolor='black')
import matplotlib.pyplot as plt
plt.title('Rating Distribution (valid only)')
plt.xlabel('Rating')
plt.ylabel('Count')

## Top 20 Organizations

In [ ]:
top_orgs = df['Organization'].value_counts().head(20)
top_orgs.plot(kind='barh', figsize=(8, 6), title='Top 20 Organizations')
import matplotlib.pyplot as plt
plt.gca().invert_yaxis()
plt.xlabel('Number of Courses')
plt.tight_layout()

## Skills Analysis

In [ ]:
import ast

empty_skills = (df['Skills'] == '[]').sum()
print(f'Empty Skills: {empty_skills} / {total} ({empty_skills/total*100:.1f}%)')
print()

# Parse and count individual skills
all_skills = []
for s in df['Skills']:
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, list):
            all_skills.extend([sk.strip() for sk in parsed])
    except:
        pass

skill_counts = pd.Series(all_skills).value_counts()
print(f'Total unique skills: {skill_counts.nunique()}')
print(f'\nTop 20 skills:')
print(skill_counts.head(20))

## Description Length

In [ ]:
desc_len = df['Description'].dropna().str.len()
print(desc_len.describe())
print()
desc_len.hist(bins=50, edgecolor='black')
import matplotlib.pyplot as plt
plt.title('Description Length Distribution')
plt.xlabel('Characters')
plt.ylabel('Count')

## Modules/Courses

In [ ]:
print(df['Modules/Courses'].value_counts(dropna=False).head(15))
print()
# Single course vs specialization (series)
is_series = df['Modules/Courses'].str.contains('series', na=False).sum()
is_module = df['Modules/Courses'].str.contains('module', na=False).sum()
print(f'Single courses (modules): {is_module}')
print(f'Specializations (series): {is_series}')

## Satisfaction Rate

In [ ]:
print(f'Available: {df["Satisfaction Rate"].notna().sum()} / {total} ({df["Satisfaction Rate"].notna().sum()/total*100:.1f}%)')
print()
print(df['Satisfaction Rate'].value_counts(dropna=False))

## Summary for RAG

Key fields for RAG/semantic search:

| Field | Usability | Notes |
|---|---|---|
| **Description** | Primary source | 99.8% available, avg 3,198 chars |
| **title** | High | 100% available |
| **Skills** | Supplementary | 29.4% empty (`[]`) |
| **Organization** | Filter/metadata | 100% available, 298 unique |
| **Level** | Filter/metadata | 88.3% available (3 levels) |
| **rating** | Filter/metadata | 78.4% valid numeric |
| **Satisfaction Rate** | Low priority | 66.9% missing |
| **enrolled** | Low priority | 26.5% placeholder |